[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChrisW09/Python-for-AI-Driven-Automation/blob/main/12_cicd/lab01_docker_and_compose.ipynb)

# 🧪 Lab 1 — Docker & Compose, hands-on

> **Module:** CI/CD & Deployment (Module 12) · **Estimated time:** ~60 minutes · **Difficulty:** Intermediate

This lab is the hands-on companion to the [Docker](docker.md) and [Docker Compose](docker-compose.md) chapters. Those chapters give you the concepts — image = **recipe**, container = **meal**, volume = **pantry**, Compose file = **menu card**. Here you get your hands on the real files we ship: the backend `Dockerfile`, its `.dockerignore`, and the four-service `docker-compose.yml` from the [example app](example-app/docker-compose.yml).

**Everything in this lab runs 100% offline, with no Docker installed.** That is not a limitation — it is the point. Instead of typing `docker build` and watching a wall of output scroll by, we *parse and simulate* what Docker does in plain Python: split a Dockerfile into instructions, hash instructions into layers, replay the build cache, match `.dockerignore` patterns, and tick through a Compose startup second by second. When you later run the real commands (on a machine with Docker Desktop), nothing will surprise you — you will have already watched every moving part in slow motion.

> 🧭 **Mental model: X-ray the kitchen.** The chapters handed you the recipe card (Dockerfile) and the menu card (Compose file) and told you what happens when the kitchen runs them. This lab is the X-ray machine: we put those exact cards under the scanner and watch — layer by layer, tick by tick — what the kitchen *would* do. A recipe is just text until you can predict what the cook does with every line; by the end of this lab, you can.

## ✅ Prerequisites

You should be comfortable with functions and dictionaries (NB 5 — functions and modules; NB 4 — dictionaries). Having read [docker.md](docker.md) and [docker-compose.md](docker-compose.md) first helps a lot — this lab deliberately reuses their terminology and examples — but every concept is briefly re-introduced as it appears.

## 🎯 Learning objectives

By the end of this lab you can:

1. **Parse a production Dockerfile** into its instructions and explain what each of `FROM`, `ENV`, `WORKDIR`, `COPY`, `RUN`, `USER`, `EXPOSE`, `HEALTHCHECK`, and `CMD` contributes to the image.
2. **Explain images as layer stacks** — why each instruction produces one cached, content-addressed layer.
3. **Predict cache hits and misses** for any edit, and justify the "copy `requirements.txt` first, code last" ordering rule.
4. **Apply `.dockerignore` patterns** to a file tree and quantify how much build context they save.
5. **Read a multi-service `docker-compose.yml`** — services, ports, `depends_on`, healthchecks, volumes — and trace who may talk to whom.
6. **Derive and simulate the startup order** Compose enforces, including why `condition: service_healthy` prevents the classic "backend crashed because the DB wasn't ready" race.

## 1. Why containers — "but it works on my machine"

The [Docker chapter](docker.md) opens with the wall every Python developer eventually hits: your FastAPI service runs perfectly on your laptop, then explodes on a colleague's machine. You have Python 3.12; they have 3.10. You installed `libpq` months ago and forgot; their `psycopg` fails to import. An environment variable from your shell profile is missing for them. Every one of these bugs has the same root cause: **software does not run in a vacuum** — it depends on a specific OS, specific system libraries, a specific runtime, and specific configuration, and only *your* machine happens to have all of them.

**Docker** packages your application *together with* everything it needs — OS libraries, the Python runtime, your dependencies, your code, and the start command — into a single, portable, immutable unit called a **container image**. Anyone with Docker can run that image and get byte-for-byte the same environment you built. The recipe analogy from the chapter: instead of handing your friend a recipe and hoping their kitchen matches yours, you ship *the whole kitchen*. Let's make the "environment drift" problem concrete with two fake machines:

In [1]:
# Two machines, same code, different environments — spot the drift.
laptop = {
    "python": "3.12.4",
    "libpq installed": True,
    "DATABASE_URL set": True,
    "os": "macOS 15",
}
colleague = {
    "python": "3.10.9",
    "libpq installed": False,
    "DATABASE_URL set": False,
    "os": "Ubuntu 22.04",
}

print(f"{'what':22} {'your laptop':12} {'colleague':12} same?")
print("-" * 58)
for key in laptop:
    same = "✅" if laptop[key] == colleague[key] else "💥"
    print(f"{key:22} {str(laptop[key]):12} {str(colleague[key]):12} {same}")

print("\nEvery 💥 row is a potential 'works on my machine' bug.")
print("A container image pins ALL of these rows at build time.")

what                   your laptop  colleague    same?
----------------------------------------------------------
python                 3.12.4       3.10.9       💥
libpq installed        True         False        💥
DATABASE_URL set       True         False        💥
os                     macOS 15     Ubuntu 22.04 💥

Every 💥 row is a potential 'works on my machine' bug.
A container image pins ALL of these rows at build time.


## 2. Dockerfile anatomy, parsed

You don't write images directly — you write a **Dockerfile** (the written recipe card) and Docker *builds* it into an image. Below we load the **real** backend Dockerfile from `example-app/backend/Dockerfile` — the exact file [docker.md §10](docker.md) dissects line by line.

The loader tries three sources in order, so this cell works anywhere: the local file (when you run this notebook inside the course repo), the course's GitHub raw URL (when you're on Colab with internet), and finally an inline copy baked into this notebook (fully offline — never breaks).

In [2]:
from pathlib import Path

# Inline fallback: a verbatim copy of example-app/backend/Dockerfile,
# so this lab still works with no repo checkout and no internet.
DOCKERFILE_FALLBACK = """\
# syntax=docker/dockerfile:1

# ---- Base image: small, official Python ----
FROM python:3.12-slim

# Sensible Python defaults inside containers
ENV PYTHONUNBUFFERED=1 \\
    PYTHONDONTWRITEBYTECODE=1

# All later paths are relative to /app
WORKDIR /app

# Copy ONLY requirements first so the pip layer is cached until deps change
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Now copy the application code
COPY app ./app

# Run as a non-root user (never run containers as root in production)
RUN useradd --create-home appuser && chown -R appuser /app
USER appuser

# Document the port the app listens on
EXPOSE 8000

# Container-level liveness check
HEALTHCHECK --interval=30s --timeout=3s --start-period=10s --retries=3 \\
  CMD python -c "import urllib.request,sys; sys.exit(0 if urllib.request.urlopen('http://localhost:8000/api/health').status==200 else 1)"

# Start the ASGI server, bound to all interfaces so Docker can reach it
CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"]
"""

RAW_BASE = ("https://raw.githubusercontent.com/ChrisW09/"
            "Python-for-AI-Driven-Automation/main/12_cicd/")


def load_text(relpath: str, fallback: str) -> tuple[str, str]:
    """Load a course file: local checkout -> GitHub raw -> inline fallback."""
    local = Path(relpath)                      # notebook runs inside 12_cicd/
    if local.exists():
        return local.read_text(), f"local file  {relpath}"
    try:
        import urllib.request
        with urllib.request.urlopen(RAW_BASE + relpath, timeout=5) as r:
            return r.read().decode(), f"GitHub raw  {relpath}"
    except Exception:
        return fallback, "inline fallback (fully offline)"


dockerfile_text, source = load_text("example-app/backend/Dockerfile", DOCKERFILE_FALLBACK)
print(f"Loaded from: {source}  ({len(dockerfile_text)} chars)\n")
print("\n".join(dockerfile_text.splitlines()[:8]) + "\n...")

Loaded from: local file  example-app/backend/Dockerfile  (1037 chars)

# syntax=docker/dockerfile:1

# ---- Base image: small, official Python ----
FROM python:3.12-slim

# Sensible Python defaults inside containers
ENV PYTHONUNBUFFERED=1 \
    PYTHONDONTWRITEBYTECODE=1
...


A Dockerfile is deliberately easy to parse: one **instruction per line** (an UPPERCASE keyword plus arguments), `#` comments, and `\` to continue a long line. Two special cases worth knowing before we write the parser:

- The very first line, `# syntax=docker/dockerfile:1`, *looks* like a comment but is a **directive** telling Docker which Dockerfile frontend to use. Harmless and recommended; keep it as line 1.
- A trailing `\` means "this instruction continues on the next line" — our `ENV` and `HEALTHCHECK` both use it.

Let's split the file into instructions and tabulate what each one does (meanings condensed from [docker.md §10](docker.md)):

In [3]:
import pandas as pd


def parse_dockerfile(text: str) -> list[dict]:
    """Split a Dockerfile into instructions, honoring \\-line-continuations."""
    # 1. stitch continued lines back together
    logical, buffer = [], ""
    for raw in text.splitlines():
        line = buffer + raw.strip()
        if line.endswith("\\"):
            buffer = line[:-1].rstrip() + " "   # drop the backslash, keep going
            continue
        logical.append(line)
        buffer = ""
    # 2. keep only real instructions (skip blanks and comments/directives)
    instructions = []
    for line in logical:
        if not line or line.startswith("#"):
            continue
        keyword, _, args = line.partition(" ")
        instructions.append({"instruction": keyword, "args": args.strip()})
    return instructions


MEANING = {
    "FROM":        "Base image — the foundation layer everything stacks on",
    "ENV":         "Set environment variables inside the image",
    "WORKDIR":     "cd for all later steps (creates the dir if needed)",
    "COPY":        "Copy files from the build context into the image",
    "RUN":         "Execute a command at BUILD time; result becomes a layer",
    "USER":        "Switch to a non-root user for everything that follows",
    "EXPOSE":      "Document the port the app listens on (docs, not publish!)",
    "HEALTHCHECK": "How Docker probes 'is this container healthy?'",
    "CMD":         "Default command when a container starts (RUN ≠ CMD!)",
}

instructions = parse_dockerfile(dockerfile_text)
df = pd.DataFrame([
    {"#": i + 1,
     "instruction": ins["instruction"],
     "args": ins["args"][:58] + ("…" if len(ins["args"]) > 58 else ""),
     "what it does": MEANING.get(ins["instruction"], "?")}
    for i, ins in enumerate(instructions)
])
df

,#,instruction,args,what it does
0,1,FROM,python:3.12-slim,Base image — the foundation layer everything s...
1,2,ENV,PYTHONUNBUFFERED=1 PYTHONDONTWRITEBYTECODE=1,Set environment variables inside the image
2,3,WORKDIR,/app,cd for all later steps (creates the dir if nee...
3,4,COPY,requirements.txt .,Copy files from the build context into the image
4,5,RUN,pip install --no-cache-dir -r requirements.txt,Execute a command at BUILD time; result become...
5,6,COPY,app ./app,Copy files from the build context into the image
6,7,RUN,useradd --create-home appuser && chown -R appu...,Execute a command at BUILD time; result become...
7,8,USER,appuser,Switch to a non-root user for everything that ...
8,9,EXPOSE,8000,"Document the port the app listens on (docs, no..."
9,10,HEALTHCHECK,--interval=30s --timeout=3s --start-period=10s...,How Docker probes 'is this container healthy?'


Read the table top to bottom and the chapter's story reappears:

- **`FROM python:3.12-slim`** — the `name:tag` of the **base image**; `-slim` is the trimmed-down Debian variant (small image, faster pulls, smaller attack surface).
- **`ENV PYTHONUNBUFFERED=1 PYTHONDONTWRITEBYTECODE=1`** — logs flush immediately (so `docker logs` shows them live) and no `.pyc` clutter.
- **`COPY requirements.txt .` before `RUN pip install …` before `COPY app ./app`** — *the* ordering trick. Dependencies change rarely, code changes constantly; §4 of this lab proves why this order matters, with numbers.
- **`USER appuser`** — containers run as **root** by default; production images switch to an unprivileged user.
- **`EXPOSE 8000`** — pure documentation. Publishing a port is a *run-time* decision (`-p 8000:8000`), not a build-time one.
- **`HEALTHCHECK`** — every 30s, hit `/api/health`; 3s timeout, 10s grace period at startup, unhealthy after 3 consecutive failures. Compose builds on exactly this in §6–7.
- **`CMD ["uvicorn", …, "--host", "0.0.0.0"]`** — the default start command. `--host 0.0.0.0` binds all interfaces; bind `127.0.0.1` instead and Docker cannot route traffic to your app — a classic trap.

⚠️ **`RUN` vs `CMD`** trips up almost everyone once: `RUN` executes **while building** the image (its *result* is baked into a layer); `CMD` only *records* what to execute **when a container starts**. A Dockerfile can have many `RUN`s but only one effective `CMD`.

---

### ✋ Quick exercise (~2 min) — Interrogate the recipe

> 🧑‍🏫 *Live checkpoint — pause and try this before peeking at the solution.*

Write a helper `find_instruction(instructions, keyword)` that returns a **list of the `args` strings** for every instruction matching `keyword` (e.g. there are two `COPY`s). Then use it to answer, with code:

1. Which port does this image document?  (`EXPOSE`)
2. What is the default start command?  (`CMD`)
3. How many `COPY` instructions are there — and why two (think back to the caching story)?

In [4]:
# ✍️ Your turn 👇
def find_instruction(instructions: list[dict], keyword: str) -> list[str]:
    # return the args of every instruction whose "instruction" == keyword
    ...

# port    = find_instruction(instructions, "EXPOSE")
# command = find_instruction(instructions, "CMD")
# copies  = find_instruction(instructions, "COPY")
# print("documented port:", port)
# print("default command:", command)
# print("COPY count:", len(copies), copies)

<details>
<summary>✅ <b>Solution</b></summary>

```python
def find_instruction(instructions: list[dict], keyword: str) -> list[str]:
    return [ins["args"] for ins in instructions if ins["instruction"] == keyword]


port    = find_instruction(instructions, "EXPOSE")
command = find_instruction(instructions, "CMD")
copies  = find_instruction(instructions, "COPY")
print("documented port:", port)
print("default command:", command)
print("COPY count:", len(copies), copies)
```

The image documents port `8000` and starts uvicorn bound to `0.0.0.0:8000`. There are **two** `COPY`s on purpose: `requirements.txt` first (changes rarely → its `pip install` layer stays cached), the `app` source last (changes constantly → only cheap layers rebuild). That split *is* the build-cache strategy we simulate next.
</details>

## 3. 🔬 Images are layer stacks

Here is the single most useful thing to know about a Docker image: it is **not** one opaque blob. It is a **stack of read-only layers**, one per instruction that changes the filesystem (`FROM`, `COPY`, `RUN`, …), each identified by a **content hash** of everything that went into it. Stack them in order and you have the image; change one layer and every layer *above* it must be rebuilt (its input — the parent — changed).

We can make this concrete without Docker. Give each instruction a layer id derived from a SHA-256 of *(its parent's id + the instruction text + any file content it copies)*. That last part matters: a `COPY` layer's identity depends on the **bytes it copies**, not just the word `COPY` — which is the whole basis of the build cache in §4.

In [5]:
import hashlib


def layer_key(parent: str, ins: dict, context: dict) -> str:
    """Content id of the layer this instruction produces (first 12 hex chars)."""
    material = parent + "|" + ins["instruction"] + " " + ins["args"]
    if ins["instruction"] == "COPY":
        src = ins["args"].split()[0]          # e.g. "requirements.txt" or "app"
        material += "|" + context.get(src, "")  # the copied bytes are part of it
    return hashlib.sha256(material.encode()).hexdigest()[:12]


# A "context" = the build-context files this Dockerfile copies, with their contents.
context_v1 = {
    "requirements.txt": "fastapi==0.111\nuvicorn==0.30\npsycopg==3.2\n",
    "app": "def app():  # v1\n    return 'hello'\n",
}

parent, chain = "scratch", []
for ins in instructions:                       # `instructions` from §2
    key = layer_key(parent, ins, context_v1)
    chain.append({"instruction": f'{ins["instruction"]} {ins["args"]}'[:34],
                  "layer": key, "built on": parent[:12]})
    parent = key

import pandas as pd
print("Bottom (FROM) to top (CMD) — each layer stacks on the one below:\n")
pd.DataFrame(chain)

Bottom (FROM) to top (CMD) — each layer stacks on the one below:



,instruction,layer,built on
0,FROM python:3.12-slim,c5c38cb0f821,scratch
1,ENV PYTHONUNBUFFERED=1 PYTHONDONTW,be5eb6fadbf9,c5c38cb0f821
2,WORKDIR /app,02f7ecb33e8c,be5eb6fadbf9
3,COPY requirements.txt .,8cf6df9d2d34,02f7ecb33e8c
4,RUN pip install --no-cache-dir -r,13c9bb193af5,8cf6df9d2d34
5,COPY app ./app,d5d17e5e2981,13c9bb193af5
6,RUN useradd --create-home appuser,b386b79beca7,d5d17e5e2981
7,USER appuser,f45b7de26213,b386b79beca7
8,EXPOSE 8000,cc32ac893a17,f45b7de26213
9,HEALTHCHECK --interval=30s --timeo,926873e13217,cc32ac893a17


Read it bottom-to-top like a stack: `FROM` is the foundation (`built on: scratch`), and every later layer names the one below it as its parent. Two files were "copied" into the context (`requirements.txt` and `app`), so the two `COPY` layers folded real bytes into their hashes. Change one byte of either and that layer's id changes — and so does every id above it. That cascade is exactly what we exploit next.

## 4. The build cache, demonstrated

When you run `docker build` a second time, Docker walks the layers bottom-to-top and reuses every layer whose id it has seen before — until it hits the **first** changed layer. From there up, everything rebuilds (each layer's parent changed, so its own id changed too). The rule in one line:

> **A layer is a cache HIT only if its content id is known *and* every layer beneath it was also a hit.**

That single rule explains the ordering trick from §2 (`COPY requirements.txt` *before* `COPY app`). Let's prove it: build once cold, then rebuild after (a) editing only the app code, and (b) editing `requirements.txt`. Watch which layers say `CACHED` vs `REBUILD`.

In [6]:
def build(instructions, context, cache: set) -> list[dict]:
    """Simulate `docker build`: reuse known layers until the first change."""
    parent, parent_rebuilt, rows = "scratch", False, []
    for ins in instructions:
        key = layer_key(parent, ins, context)
        hit = (key in cache) and not parent_rebuilt   # <-- the rule
        rows.append({"step": f'{ins["instruction"]} {ins["args"]}'[:34],
                     "status": "CACHED ✅" if hit else "REBUILD 🔨"})
        cache.add(key)
        parent, parent_rebuilt = key, parent_rebuilt or (not hit)
    return rows


cache = set()
cold = build(instructions, context_v1, cache)          # first ever build
print("BUILD 1 — cold cache (everything is new):")
print(pd.DataFrame(cold).to_string(index=False), "\n")

# (a) change ONLY the application code
context_edit_code = dict(context_v1, app="def app():  # v2 — new feature!\n    return 'hi'\n")
warm = build(instructions, context_edit_code, cache)
print("BUILD 2 — after editing app code only:")
print(pd.DataFrame(warm).to_string(index=False))

BUILD 1 — cold cache (everything is new):
                              step    status
             FROM python:3.12-slim REBUILD 🔨
ENV PYTHONUNBUFFERED=1 PYTHONDONTW REBUILD 🔨
                      WORKDIR /app REBUILD 🔨
           COPY requirements.txt . REBUILD 🔨
RUN pip install --no-cache-dir -r  REBUILD 🔨
                    COPY app ./app REBUILD 🔨
RUN useradd --create-home appuser  REBUILD 🔨
                      USER appuser REBUILD 🔨
                       EXPOSE 8000 REBUILD 🔨
HEALTHCHECK --interval=30s --timeo REBUILD 🔨
CMD ["uvicorn", "app.main:app", "- REBUILD 🔨 

BUILD 2 — after editing app code only:
                              step    status
             FROM python:3.12-slim  CACHED ✅
ENV PYTHONUNBUFFERED=1 PYTHONDONTW  CACHED ✅
                      WORKDIR /app  CACHED ✅
           COPY requirements.txt .  CACHED ✅
RUN pip install --no-cache-dir -r   CACHED ✅
                    COPY app ./app REBUILD 🔨
RUN useradd --create-home appuser  REBUILD 🔨
                 

Look at BUILD 2: everything up to and including `RUN pip install …` says **CACHED** — the expensive dependency install was skipped entirely — and only `COPY app ./app` and the cheap layers above it rebuilt. That is seconds instead of minutes on every code change.

Now flip it: edit `requirements.txt` instead and rebuild. Because that `COPY` sits *below* the `pip install`, the cascade starts earlier — the slow install layer is invalidated too.

In [7]:
# (b) change requirements.txt — a lower layer, so more rebuilds
context_edit_deps = dict(context_v1, **{"requirements.txt": "fastapi==0.111\nuvicorn==0.30\npsycopg==3.2\nredis==5.0\n"})
deps = build(instructions, context_edit_deps, cache)
print("BUILD 3 — after adding one line to requirements.txt:")
print(pd.DataFrame(deps).to_string(index=False))
print("\nThe pip-install layer REBUILT this time — that's the minutes-long one.")
print("Lesson: put what changes RARELY (deps) below what changes OFTEN (code).")

BUILD 3 — after adding one line to requirements.txt:
                              step    status
             FROM python:3.12-slim  CACHED ✅
ENV PYTHONUNBUFFERED=1 PYTHONDONTW  CACHED ✅
                      WORKDIR /app  CACHED ✅
           COPY requirements.txt . REBUILD 🔨
RUN pip install --no-cache-dir -r  REBUILD 🔨
                    COPY app ./app REBUILD 🔨
RUN useradd --create-home appuser  REBUILD 🔨
                      USER appuser REBUILD 🔨
                       EXPOSE 8000 REBUILD 🔨
HEALTHCHECK --interval=30s --timeo REBUILD 🔨
CMD ["uvicorn", "app.main:app", "- REBUILD 🔨

The pip-install layer REBUILT this time — that's the minutes-long one.
Lesson: put what changes RARELY (deps) below what changes OFTEN (code).


---

### ✋ Quick exercise (~2 min) — Count the wasted work

> 🧑‍🏫 *Live checkpoint — pause and try this before peeking at the solution.*

The `status` strings end in `✅` (cached) or `🔨` (rebuilt). Write `count_rebuilds(rows)` that returns how many layers rebuilt in a build, then compare the *code-only* edit against the *requirements* edit:

1. How many layers rebuilt for the app-code change (`warm`)?
2. How many for the requirements change (`deps`)?
3. In one sentence: why is the second number bigger?

In [8]:
# ✍️ Your turn 👇
def count_rebuilds(rows: list[dict]) -> int:
    # count rows whose "status" indicates a REBUILD (ends with 🔨)
    ...

# print("code edit rebuilt:", count_rebuilds(warm))
# print("deps edit rebuilt:", count_rebuilds(deps))

<details>
<summary>✅ <b>Solution</b></summary>

```python
def count_rebuilds(rows: list[dict]) -> int:
    return sum(1 for r in rows if r["status"].endswith("🔨"))


print("code edit rebuilt:", count_rebuilds(warm))
print("deps edit rebuilt:", count_rebuilds(deps))
```

Editing the app code rebuilds only the `COPY app` layer and the two cheap layers above it. Editing `requirements.txt` invalidates a layer *lower* in the stack, so the pip-install layer and everything above it rebuild too — more layers, and the slow one among them. That asymmetry is the entire reason for the `requirements`-first ordering.
</details>

## 5. `.dockerignore` — shrink the build context

Before Docker builds anything, it packs up the **build context** — the directory you point `docker build` at — and sends it to the Docker daemon. Ship your `.venv/`, `__pycache__/`, and `.git/` along and that tarball balloons to hundreds of MB, every build gets slower, and secrets can leak into the image. A **`.dockerignore`** file (same idea as `.gitignore`) excludes paths from the context.

We load the **real** `example-app/backend/.dockerignore`, then apply it to a sample file tree with a small `fnmatch`-based matcher. (Docker's real matching has more edge cases — this captures the common ones.)

In [9]:
DOCKERIGNORE_FALLBACK = "__pycache__/\n*.pyc\n.pytest_cache/\ntests/\n.venv/\n"

dockerignore_text, source = load_text("example-app/backend/.dockerignore", DOCKERIGNORE_FALLBACK)
patterns = [ln.strip() for ln in dockerignore_text.splitlines() if ln.strip() and not ln.startswith("#")]
print(f"Loaded from: {source}")
print("Ignore patterns:", patterns)

Loaded from: local file  example-app/backend/.dockerignore
Ignore patterns: ['__pycache__/', '*.pyc', '.pytest_cache/', 'tests/', '.venv/']


In [10]:
import fnmatch


def is_ignored(path: str, patterns: list[str]) -> bool:
    """True if `path` matches any .dockerignore pattern (simplified rules)."""
    parts = path.split("/")
    for pat in patterns:
        p = pat.rstrip("/")                       # "tests/" -> "tests"
        if fnmatch.fnmatch(path, p):              # whole-path glob, e.g. "*.pyc"
            return True
        if any(fnmatch.fnmatch(seg, p) for seg in parts):  # a dir/file component
            return True
    return False


# A realistic backend build context, each file with a size in KB.
build_context = {
    "app/main.py": 12,
    "app/__pycache__/main.cpython-312.pyc": 34,
    "requirements.txt": 1,
    "Dockerfile": 1,
    "tests/test_main.py": 8,
    "tests/__pycache__/test_main.cpython-312.pyc": 30,
    ".pytest_cache/v/cache/lastfailed": 2,
    ".venv/lib/python3.12/site-packages/fastapi/__init__.py": 900,
    "app/models.pyc": 5,
}

rows = [{"path": p, "KB": kb, "sent?": "🚫 ignored" if is_ignored(p, patterns) else "📦 sent"}
        for p, kb in build_context.items()]
sent = sum(kb for p, kb in build_context.items() if not is_ignored(p, patterns))
total = sum(build_context.values())
print(pd.DataFrame(rows).to_string(index=False))
print(f"\nContext size: {total} KB total  ->  {sent} KB sent  ({total - sent} KB saved, {100*(total-sent)//total}%)")

                                                  path  KB     sent?
                                           app/main.py  12    📦 sent
                  app/__pycache__/main.cpython-312.pyc  34 🚫 ignored
                                      requirements.txt   1    📦 sent
                                            Dockerfile   1    📦 sent
                                    tests/test_main.py   8 🚫 ignored
           tests/__pycache__/test_main.cpython-312.pyc  30 🚫 ignored
                      .pytest_cache/v/cache/lastfailed   2 🚫 ignored
.venv/lib/python3.12/site-packages/fastapi/__init__.py 900 🚫 ignored
                                        app/models.pyc   5 🚫 ignored

Context size: 993 KB total  ->  14 KB sent  (979 KB saved, 98%)


The `.venv/` line alone saved 900 KB in this toy tree — on a real project it is the difference between a 2 MB context and a 400 MB one. Note *why* each row is excluded: `*.pyc` matches by whole-path glob, while `tests` and `.venv` match as **path components** (any directory of that name, at any depth). Everything Docker actually needs — `app/main.py`, `requirements.txt`, the `Dockerfile` — still ships.

---

### ✋ Quick exercise (~2 min) — Tighten the ignore file

> 🧑‍🏫 *Live checkpoint — pause and try this before peeking at the solution.*

Your team wants to also keep **`.env` secret files** and **`*.log`** files out of every image. Build `patterns_plus` by adding those two patterns to `patterns`, then re-run the matcher on this extended tree and print which paths are newly excluded:

```python
extra_tree = ["app/main.py", ".env", "app/debug.log", "config/prod.env"]
```

In [11]:
# ✍️ Your turn 👇
extra_tree = ["app/main.py", ".env", "app/debug.log", "config/prod.env"]
patterns_plus = ...   # patterns + two new ones: ".env" and "*.log"

# for p in extra_tree:
#     print(p, "->", "ignored" if is_ignored(p, patterns_plus) else "sent")

<details>
<summary>✅ <b>Solution</b></summary>

```python
extra_tree = ["app/main.py", ".env", "app/debug.log", "config/prod.env"]
patterns_plus = patterns + [".env", "*.log"]

for p in extra_tree:
    print(p, "->", "ignored" if is_ignored(p, patterns_plus) else "sent")
```

`app/main.py` still ships; `.env` matches the literal `.env` component; `app/debug.log` matches `*.log`. Note that `config/prod.env` is **not** caught by a bare `.env` pattern — matching secrets reliably usually needs a broader glob like `*.env`, a good reminder that an over-narrow ignore rule gives false confidence.
</details>

## 6. Compose: the whole meal

One image is one dish. A real app is a **meal** — our example app is four services: a FastAPI `backend`, a static `frontend`, a Postgres `db`, and an `nginx` reverse proxy. A **`docker-compose.yml`** file declares them all, how they connect, and in what order they start. We load the **real** four-service file we ship and parse it.

In [12]:
COMPOSE_FALLBACK = """\
services:
  backend:
    build: ./backend
    image: ghcr.io/chrisw09/example-app-backend:latest
    environment:
      DATABASE_URL: postgresql://app:app@db:5432/app
    depends_on:
      db:
        condition: service_healthy
    restart: unless-stopped
    networks: [appnet]
  frontend:
    build: ./frontend
    image: ghcr.io/chrisw09/example-app-frontend:latest
    restart: unless-stopped
    networks: [appnet]
  db:
    image: postgres:16-alpine
    environment:
      POSTGRES_USER: app
      POSTGRES_PASSWORD: app
      POSTGRES_DB: app
    volumes:
      - db_data:/var/lib/postgresql/data
    healthcheck:
      test: ["CMD-SHELL", "pg_isready -U app"]
      interval: 10s
      timeout: 5s
      retries: 5
    restart: unless-stopped
    networks: [appnet]
  nginx:
    build: ./nginx
    image: ghcr.io/chrisw09/example-app-nginx:latest
    ports:
      - "80:80"
    depends_on:
      - backend
      - frontend
    restart: unless-stopped
    networks: [appnet]
volumes:
  db_data:
networks:
  appnet:
"""

# Pre-parsed fallback, used only if PyYAML isn't installed.
COMPOSE_DICT = {
    "services": {
        "backend":  {"depends_on": {"db": {"condition": "service_healthy"}},
                     "environment": {"DATABASE_URL": "postgresql://app:app@db:5432/app"},
                     "networks": ["appnet"]},
        "frontend": {"networks": ["appnet"]},
        "db":       {"image": "postgres:16-alpine",
                     "volumes": ["db_data:/var/lib/postgresql/data"],
                     "healthcheck": {"test": ["CMD-SHELL", "pg_isready -U app"]},
                     "networks": ["appnet"]},
        "nginx":    {"ports": ["80:80"], "depends_on": ["backend", "frontend"],
                     "networks": ["appnet"]},
    },
    "volumes": {"db_data": None},
    "networks": {"appnet": None},
}

compose_text, source = load_text("example-app/docker-compose.yml", COMPOSE_FALLBACK)
try:
    import yaml
    compose = yaml.safe_load(compose_text)
    parsed_via = "yaml.safe_load"
except Exception:
    compose, parsed_via = COMPOSE_DICT, "inline dict fallback (PyYAML not installed)"

services = compose["services"]
print(f"Loaded from: {source}")
print(f"Parsed via:  {parsed_via}")
print(f"Services:    {list(services)}")

Loaded from: local file  example-app/docker-compose.yml
Parsed via:  yaml.safe_load
Services:    ['backend', 'frontend', 'db', 'nginx']


In [13]:
def dep_names(svc: dict) -> list[str]:
    """depends_on can be a LIST [a, b] or a DICT {a: {condition: ...}} — handle both."""
    dep = svc.get("depends_on") or []
    return list(dep.keys()) if isinstance(dep, dict) else list(dep)


overview = []
for name, svc in services.items():
    overview.append({
        "service": name,
        "publishes port": ", ".join(svc.get("ports", [])) or "—",
        "depends_on": ", ".join(dep_names(svc)) or "—",
        "healthcheck": "yes" if "healthcheck" in svc else "—",
        "volumes": ", ".join(svc.get("volumes", [])) or "—",
    })
pd.DataFrame(overview)

,service,publishes port,depends_on,healthcheck,volumes
0,backend,—,db,—,—
1,frontend,—,—,—,—
2,db,—,—,yes,db_data:/var/lib/postgresql/data
3,nginx,80:80,"backend, frontend",—,—


Everything the chapter describes is right there in the table:

- Only **nginx publishes a port** (`80:80`) — it is the single front door. The backend and db are reachable *inside* the private `appnet` network but never exposed to the host directly. (Security win: your database is not on the public internet.)
- **`db` has a healthcheck** (`pg_isready`) and **`backend` depends_on db** — so Compose can wait for the database to be *actually ready*, not merely started. That is the race we simulate next.
- **`db` has a named volume** (`db_data`) — the pantry that survives `docker compose down`, so your data outlives the container.

## 7. 🔬 Startup order, simulated

`depends_on` controls **order**, but there are two strengths of it, and the difference is the source of a classic production bug:

- **list form** (`depends_on: [backend, frontend]`, used by nginx) waits only until the dependency has *started* — the process exists, nothing more.
- **`condition: service_healthy`** (used by backend → db) waits until the dependency's **healthcheck passes**.

Why it matters: Postgres takes a few seconds to accept connections *after* its process starts. If the backend only waited for "started", it would launch too early, fail to connect, and crash. Let's build the dependency graph, topologically sort it, then tick through startup — a service becomes `healthy` a few ticks after it `starts`, and a `service_healthy` dependant refuses to start until then.

In [14]:
# 1. Build the depends_on graph and topologically sort it (Kahn's algorithm).
graph = {name: dep_names(svc) for name, svc in services.items()}
indeg = {n: 0 for n in graph}
for n, deps in graph.items():
    for _ in deps:
        indeg[n] += 1

order, ready = [], sorted([n for n, d in indeg.items() if d == 0])
while ready:
    n = ready.pop(0)
    order.append(n)
    for m, deps in graph.items():
        if n in deps:
            indeg[m] -= 1
            if indeg[m] == 0:
                ready.append(m)
    ready.sort()

print("A valid startup order (dependencies first):")
print("   " + "  ->  ".join(order))

A valid startup order (dependencies first):
   db  ->  backend  ->  frontend  ->  nginx


In [15]:
# 2. Tick simulation. A service may START once every dependency it needs is
#    satisfied; "service_healthy" deps must be HEALTHY, plain deps just STARTED.
HEALTH_DELAY = {"db": 3, "backend": 2, "frontend": 1, "nginx": 1}  # ticks start->healthy


def needs_healthy(svc, dep):
    """Does `svc` require `dep` to be *healthy* (not just started)?"""
    d = svc.get("depends_on")
    return isinstance(d, dict) and d.get(dep, {}).get("condition") == "service_healthy"


started, healthy_at, timeline = {}, {}, []
for tick in range(0, 9):
    for name in order:
        if name in started:
            continue
        deps = graph[name]
        ok = all((healthy_at.get(d, 1e9) <= tick) if needs_healthy(services[name], d)
                 else (d in started) for d in deps)
        if ok:
            started[name] = tick
            healthy_at[name] = tick + HEALTH_DELAY[name]
            timeline.append({"tick": tick, "event": f"{name} STARTED (healthy at tick {healthy_at[name]})"})

print(pd.DataFrame(timeline).to_string(index=False))
print("\nNote: backend waits for db to be HEALTHY (tick 3), not just started (tick 0).")

 tick                                event
    0       db STARTED (healthy at tick 3)
    0 frontend STARTED (healthy at tick 1)
    3  backend STARTED (healthy at tick 5)
    3    nginx STARTED (healthy at tick 4)

Note: backend waits for db to be HEALTHY (tick 3), not just started (tick 0).


---

### ✋ Quick exercise (~2 min) — Who can start at tick 0?

> 🧑‍🏫 *Live checkpoint — pause and try this before peeking at the solution.*

A service can start immediately (tick 0) only if it has **no** dependencies. Write `roots(graph)` returning the sorted list of such services, and confirm it matches what the simulation started first. Then answer: if you deleted the `db` healthcheck and changed backend to the plain list form, what tick would `backend` start at?

In [16]:
# ✍️ Your turn 👇
def roots(graph: dict) -> list[str]:
    # services whose dependency list is empty, sorted alphabetically
    ...

# print("can start at tick 0:", roots(graph))

<details>
<summary>✅ <b>Solution</b></summary>

```python
def roots(graph: dict) -> list[str]:
    return sorted(n for n, deps in graph.items() if not deps)


print("can start at tick 0:", roots(graph))
```

`db` and `frontend` have no dependencies, so they start at tick 0. If backend depended on db by the plain list form instead of `service_healthy`, it would start at **tick 0 too** — and then likely crash, because Postgres isn't accepting connections yet. That crash-loop is precisely what `condition: service_healthy` exists to prevent.
</details>

## 8. ⚠️ Pitfalls to avoid

The chapters end with hard-won warnings. The four that bite most often:

1. **`FROM python:latest`** — `latest` is a moving target; a rebuild months later can silently pull a new major version and break everything. **Pin a version** (`python:3.12-slim`), exactly as our Dockerfile does.
2. **Secrets baked into layers.** `ENV API_KEY=...` or `COPY .env` writes the secret into an image layer *forever* — anyone who pulls the image can read it, even if a later layer "deletes" it. Pass secrets at **run time** (env vars, secret mounts), never build time.
3. **Giant build contexts.** No `.dockerignore` → your `.venv/` and `.git/` get shipped on every build. Slow, and a leak risk. (§5.)
4. **Assuming start order = ready order.** Without `condition: service_healthy`, your app races its database and loses. (§7.)

Below, a tiny linter flags two of these in a suspicious Dockerfile — the kind of check a real CI pipeline runs (Module 12's `lab02_ci_pipeline_github_actions.ipynb` builds exactly that).

In [17]:
suspicious = [
    "FROM python:latest",
    "ENV SECRET_API_KEY=sk-prod-12345",
    "COPY . .",
    'CMD ["python", "app.py"]',
]


def lint_dockerfile(lines: list[str]) -> list[str]:
    warnings = []
    for ln in lines:
        if ln.startswith("FROM") and (":latest" in ln or ":" not in ln.split()[-1]):
            warnings.append(f"⚠️  unpinned base image → {ln!r}")
        if ln.startswith("ENV") and any(s in ln.upper() for s in ("KEY", "SECRET", "TOKEN", "PASSWORD")):
            warnings.append(f"⚠️  possible secret baked into a layer → {ln!r}")
    return warnings


for w in lint_dockerfile(suspicious):
    print(w)
print("\n2 issues found — both would pass `docker build` and bite you later.")

⚠️  unpinned base image → 'FROM python:latest'
⚠️  possible secret baked into a layer → 'ENV SECRET_API_KEY=sk-prod-12345'

2 issues found — both would pass `docker build` and bite you later.


### Is real Docker around?

Nothing above needed it — but if you *do* have Docker installed, here is the one command that turns everything you simulated into the real thing. This cell only reports; it never runs a build.

In [18]:
import shutil, subprocess

if shutil.which("docker"):
    out = subprocess.run(["docker", "--version"], capture_output=True, text=True)
    print("Docker is installed:", out.stdout.strip())
    print("Try it for real:  cd example-app && docker compose up --build")
else:
    print("Docker is not installed — and you didn't need it. 🎉")
    print("Everything above ran in pure Python. Install Docker Desktop later")
    print("and `docker compose up --build` in example-app/ runs the real four services.")

Docker is not installed — and you didn't need it. 🎉
Everything above ran in pure Python. Install Docker Desktop later
and `docker compose up --build` in example-app/ runs the real four services.


## 🧪 Practice exercises

Work these after the walk-through. Each ships a collapsible solution — try first, then check.

### Exercise 1 — ⭐ Count the layers that change the filesystem

Not every instruction makes a layer you'd think of as "filesystem state." `FROM`, `COPY`, and `RUN` clearly do; `ENV`, `WORKDIR`, `EXPOSE`, `USER`, `HEALTHCHECK`, and `CMD` mostly set *metadata*. Using `instructions` from §2, count how many instructions are of the "filesystem" kind (`FROM`, `COPY`, `RUN`).

In [19]:
# your code here
FS_INSTRUCTIONS = {"FROM", "COPY", "RUN"}
# fs_layers = ...
# print(fs_layers)

<details>
<summary>✅ <b>Solution</b></summary>

```python
FS_INSTRUCTIONS = {"FROM", "COPY", "RUN"}
fs_layers = sum(1 for ins in instructions if ins["instruction"] in FS_INSTRUCTIONS)
print("filesystem layers:", fs_layers)
```

There are four (`FROM`, `COPY requirements`, `RUN pip install`, `COPY app`, `RUN useradd`) — the ones whose content genuinely changes the image filesystem and therefore dominate build time and image size.
</details>

### Exercise 2 — ⭐ Which services are reachable from the host?

Using `services` from §6, list every service that publishes at least one port (has a non-empty `ports:` list). These are the only services the outside world can reach directly.

In [20]:
# your code here
# host_reachable = ...
# print(host_reachable)

<details>
<summary>✅ <b>Solution</b></summary>

```python
host_reachable = [name for name, svc in services.items() if svc.get("ports")]
print("reachable from host:", host_reachable)
```

Only `nginx` — the single front door on port 80. `backend`, `frontend`, and `db` are reachable *inside* `appnet` but never exposed to the host, which is exactly the security posture you want.
</details>

### Exercise 3 — ⭐⭐ Quantify the `.dockerignore` win on a bigger tree

Add a `.git/` directory (say 5 000 KB across a few files) and a second `.venv` file to `build_context`, then report the **percentage** of context size that `.dockerignore` strips — but first add `.git/` to your patterns. Reuse `is_ignored` and `patterns` from §5.

In [21]:
# your code here
big_context = dict(build_context)
big_context[".git/objects/pack/pack-abc.pack"] = 5000
big_context[".venv/lib/python3.12/site-packages/numpy/core.py"] = 1200
# patterns_git = ...
# compute sent vs total and print the % saved

<details>
<summary>✅ <b>Solution</b></summary>

```python
big_context = dict(build_context)
big_context[".git/objects/pack/pack-abc.pack"] = 5000
big_context[".venv/lib/python3.12/site-packages/numpy/core.py"] = 1200
patterns_git = patterns + [".git"]

total = sum(big_context.values())
sent = sum(kb for p, kb in big_context.items() if not is_ignored(p, patterns_git))
print(f"{total} KB total -> {sent} KB sent ({100*(total-sent)//total}% stripped)")
```

With `.git/` and a fat `.venv/` in the tree, the ignore file strips the large majority of the context — turning a slow, leaky build into a fast, clean one.
</details>

### Exercise 4 — ⭐⭐ Debug me 🐞

This `service_ports` helper is meant to collect every published port across the compose file. It crashes on our real file. **Run it, read the traceback, and fix it** so it returns `["80:80"]`.

*Hint: what type is `depends_on` for nginx vs backend — and is `ports` always present?*

In [22]:
# 🐞 Buggy — run it, watch it break, then fix it.
def service_ports(services: dict) -> list[str]:
    found = []
    for name, svc in services.items():
        for port in svc["ports"]:      # bug: not every service HAS "ports"
            found.append(port)
    return found


print(service_ports(services))

KeyError: 'ports'

<details>
<summary>✅ <b>Solution</b></summary>

```python
def service_ports(services: dict) -> list[str]:
    found = []
    for name, svc in services.items():
        for port in svc.get("ports", []):   # .get with a default — no KeyError
            found.append(port)
    return found


print(service_ports(services))   # -> ['80:80']
```

The bug is `svc["ports"]`: only `nginx` declares `ports`, so the first service without it raises `KeyError`. Using `svc.get("ports", [])` treats "no ports" as an empty list — the same defensive-dict habit from NB 4. This is the single most common error when walking heterogeneous config dicts.
</details>

## 🧠 Stretch exercises

Harder, open-ended. Solutions sketch the approach rather than hand you the code.

### Stretch exercise A — ⭐⭐⭐ Simulate a multi-stage build

Real production Dockerfiles use **multi-stage builds**: a first `FROM … AS builder` stage compiles/installs everything, then a second `FROM` copies only the finished artifacts, leaving build tools behind. Extend `parse_dockerfile` to detect `AS <name>` on `FROM` lines and `--from=<stage>` on `COPY` lines, and report how many stages a Dockerfile has and which files cross between them.

<details>
<summary>💡 Approach</summary>

Track a `stage` counter that increments on each `FROM`; tag every instruction with its stage. For `COPY --from=builder`, record an edge `(builder -> current stage, path)`. The payoff to explain: only the *final* stage's layers ship, so build-only dependencies (compilers, dev headers) never reach production — often shrinking an image by 10× or more.
</details>

### Stretch exercise B — ⭐⭐⭐ Detect a dependency cycle

Our Kahn topological sort in §7 silently produces a short `order` list if the graph has a cycle (e.g. `a depends_on b` and `b depends_on a`). Add a check: if `len(order) < len(graph)`, the leftover services form a cycle — report them clearly instead of starting a broken subset.

<details>
<summary>💡 Approach</summary>

After the Kahn loop, compute `set(graph) - set(order)`. If non-empty, those nodes never reached in-degree 0 — they are in (or downstream of) a cycle. Compose itself refuses to start such a configuration; mirror that by raising a `ValueError` naming the offending services. Test it by adding a fake `db depends_on nginx` edge.
</details>

### Stretch exercise C — ⭐⭐⭐ A healthcheck state machine

Model a container's health as a state machine over time: `starting` during the `start-period`, then `healthy`/`unhealthy` based on consecutive probe results, using our Dockerfile's real values (`interval=30s`, `timeout=3s`, `start-period=10s`, `retries=3`). Feed it a list of probe outcomes and print the state at each interval.

<details>
<summary>💡 Approach</summary>

Track `consecutive_failures`. While `t < start_period`, state is `starting` (failures don't count yet). After that, each failing probe increments the counter; `retries` consecutive failures flips state to `unhealthy`; any success resets to `healthy`. This is exactly the logic Compose's `condition: service_healthy` polls — you're rebuilding the thing §7 depended on.
</details>

## 🎁 Bonus mini-project — a mini `docker build` simulator

Tie §2–§4 together into one function `docker_build(dockerfile_text, context, cache)` that:

1. parses the Dockerfile into instructions,
2. walks the layers, using `cache` to decide CACHED vs REBUILD per layer,
3. prints a build log that looks like Docker's (`Step 3/9 : RUN pip install …  --> CACHED`),
4. returns the final image id (the top layer's key) and the set of layers now cached.

Then call it three times — cold, after a code edit, after a deps edit — and confirm the final image id changes **only** when something actually changed.

<details>
<summary>💡 Solution sketch</summary>

```python
def docker_build(dockerfile_text, context, cache):
    ins = parse_dockerfile(dockerfile_text)
    parent, parent_rebuilt = "scratch", False
    for i, step in enumerate(ins, 1):
        key = layer_key(parent, step, context)
        hit = key in cache and not parent_rebuilt
        tag = "CACHED" if hit else "BUILD "
        print(f"Step {i}/{len(ins)} : {step['instruction']} {step['args'][:32]:32}  --> {tag} {key}")
        cache.add(key)
        parent, parent_rebuilt = key, parent_rebuilt or not hit
    print("Successfully built", parent)
    return parent, cache


cache = set()
img1, cache = docker_build(dockerfile_text, context_v1, cache)
img2, cache = docker_build(dockerfile_text, context_edit_code, cache)
assert img1 != img2          # code changed -> new image id
img3, cache = docker_build(dockerfile_text, context_v1, cache)
assert img3 == img1          # identical inputs -> identical id (fully cached)
```

The two assertions are the whole point: an image id is a pure function of its inputs. Same recipe + same context → same id, every time, on every machine. *That* is reproducibility.
</details>

## 🧠 Key takeaways

- **An image is a stack of content-addressed layers**, one per filesystem-changing instruction. Change a layer and everything above it rebuilds.
- **The build cache follows one rule**: a layer is reused only if its content id is known *and* every layer below it was reused. That is why you `COPY requirements.txt` and `pip install` **before** `COPY app` — deps change rarely, code changes constantly.
- **`.dockerignore` shrinks the build context**, making builds faster and keeping `.venv/`, `.git/`, and secrets out of your images.
- **A Compose file declares the whole meal** — services, ports, volumes, and dependencies. Publish only your front door; keep databases on the private network.
- **`depends_on` orders startup, but only `condition: service_healthy` waits for *ready***. Without it, your app races its database and crash-loops.
- **You simulated all of it in pure Python** — so when you run the real `docker build` and `docker compose up`, every line of output will already make sense.

## ✅ Self-assessment

- [ ] I can read a Dockerfile top to bottom and say what each instruction contributes.
- [ ] I can explain why images are layered and predict which layers a given edit rebuilds.
- [ ] I can justify the "requirements first, code last" ordering with the caching rule.
- [ ] I can write a `.dockerignore` and explain what it keeps out of the build context and why.
- [ ] I can read a `docker-compose.yml` and trace which services talk to whom and which are exposed.
- [ ] I can explain the difference between `depends_on: [x]` and `condition: service_healthy`, and the bug the latter prevents.

## 🚀 Next step

You can now read the two files that define *how software ships* — a `Dockerfile` and a `docker-compose.yml` — and predict exactly what Docker does with them. Next:

- **`lab02_ci_pipeline_github_actions.ipynb`** — build a miniature CI pipeline that lints, tests, and builds an image on every push (the automated factory from [github-actions.md](github-actions.md)).
- **Run the real thing.** On any machine with Docker Desktop: `cd example-app && docker compose up --build` starts the exact four services you dissected here. Read [docker.md](docker.md) and [docker-compose.md](docker-compose.md) alongside it.

> 🐳 You didn't install Docker to understand Docker. That's the difference between memorising commands and knowing what they *do*.